In [5]:
import sagemaker
from sagemaker import get_execution_role
import boto3
import os
from sagemaker.session import Session
from sagemaker.feature_store.feature_group import FeatureGroup  # For manipulating filepath names
import numpy as np
import pandas as pd


bucket = sagemaker.Session().default_bucket()
prefix = 'sagemaker/ChurnPredictor-linlearn'
# Define IAM role
role = get_execution_role()

In [6]:
bucket

'sagemaker-us-east-2-806700449556'

# Import the data from SageMaker Feature Store

In [7]:
region = boto3.Session().region_name
boto_session = boto3.Session(region_name=region)

sagemaker_client = boto_session.client(service_name='sagemaker', region_name=region)
featurestore_runtime = boto_session.client(service_name='sagemaker-featurestore-runtime', region_name=region)

feature_store_session = Session(
    boto_session=boto_session,
    sagemaker_client=sagemaker_client,
    sagemaker_featurestore_runtime_client=featurestore_runtime
)

feature_group_name = "FG-Churn_Predictor-8b5ad8a1"  # replace with your feature group name 
feature_group = FeatureGroup(name=feature_group_name, sagemaker_session=feature_store_session)

In [8]:
feature_group

FeatureGroup(name='FG-Churn_Predictor-8b5ad8a1', sagemaker_session=<sagemaker.session.Session object at 0x7f471367f620>, feature_definitions=[])

In [9]:
# Build SQL query to features group
fs_query = feature_group.athena_query()
fs_table = fs_query.table_name
query_string = 'SELECT * FROM "'+fs_table+'"'
print('Running ' + query_string)

Running SELECT * FROM "fg_churn_predictor_8b5ad8a1_1762656713"


In [19]:
# Run Athena query. The output is loaded to a Pandas dataframe.
fs_query.run(query_string=query_string, output_location='s3://'+bucket+'/'+prefix+'/fs_query_results/')
fs_query.wait()
model_data = fs_query.as_dataframe()

In [20]:
model_data

,custid,retained,created,firstorder,lastorder,esent,eopenrate,eclickrate,avgorder,ordfreq,...,favday_sunday,created_year,created_month,firstorder_year,firstorder_month,lastorder_year,lastorder_month,write_time,api_invocation_time,is_deleted


In [21]:
# Get transformed data from S3 since Sagemaker Prepocessing job to upload data to Feature Store has resource quota limitation.
if model_data.empty:
    print("model_data is empty. Loading data from S3 CSV file.")
    s3_csv_path = f's3://{bucket}/ChurnPredictor_Transformed_Dataset.csv'
    model_data = pd.read_csv(s3_csv_path)
    print("Data loaded successfully from S3 CSV.")
    model_data

model_data is empty. Loading data from S3 CSV file.
Data loaded successfully from S3 CSV.


In [22]:
model_data.head()

,custid,retained,created,firstorder,lastorder,esent,eopenrate,eclickrate,avgorder,ordfreq,...,favday_Thursday,favday_Wednesday,favday_Saturday,favday_Sunday,created_year,created_month,firstorder_year,firstorder_month,lastorder_year,lastorder_month
0,6H6T6N,0,2012-09-28,2013-08-11,2013-08-11,29,100.000000,3.448276,14.52,0.000000,...,0.0,0.0,0.0,0.0,2012,8,2013,7,2013,7
1,APCENR,1,2010-12-19,2011-04-01,2014-01-19,95,92.631579,10.526316,83.69,0.181641,...,0.0,0.0,0.0,0.0,2010,11,2011,3,2014,0
2,7UP6MS,0,2010-10-03,2010-12-01,2011-07-06,0,0.000000,0.000000,33.58,0.059908,...,0.0,1.0,0.0,0.0,2010,9,2010,11,2011,6
3,7ZEW8G,0,2010-10-22,2011-03-28,2011-03-28,0,0.000000,0.000000,54.96,0.000000,...,1.0,0.0,0.0,0.0,2010,9,2011,2,2011,2
4,8V726M,1,2010-11-27,2010-11-29,2013-01-28,30,90.000000,13.333333,111.91,0.008850,...,0.0,0.0,0.0,0.0,2010,10,2010,10,2013,0


In [23]:
#remove unwanted columns
model_data = model_data.drop(['created', 'firstorder', 'lastorder','custid'], axis=1) #clean up from transformation
#model_data = model_data.drop(['write_time', 'api_invocation_time', 'is_deleted'], axis=1)  # Columns added by Sagemaker Processing Job while uploading data

In [24]:
model_data.head()

,retained,esent,eopenrate,eclickrate,avgorder,ordfreq,paperless,refill,doorstep,city_BOM,...,favday_Thursday,favday_Wednesday,favday_Saturday,favday_Sunday,created_year,created_month,firstorder_year,firstorder_month,lastorder_year,lastorder_month
0,0,29,100.000000,3.448276,14.52,0.000000,0,0,0,0.0,...,0.0,0.0,0.0,0.0,2012,8,2013,7,2013,7
1,1,95,92.631579,10.526316,83.69,0.181641,1,1,1,0.0,...,0.0,0.0,0.0,0.0,2010,11,2011,3,2014,0
2,0,0,0.000000,0.000000,33.58,0.059908,0,0,0,0.0,...,0.0,1.0,0.0,0.0,2010,9,2010,11,2011,6
3,0,0,0.000000,0.000000,54.96,0.000000,0,0,0,1.0,...,1.0,0.0,0.0,0.0,2010,9,2011,2,2011,2
4,1,30,90.000000,13.333333,111.91,0.008850,0,0,0,1.0,...,0.0,0.0,0.0,0.0,2010,10,2010,10,2013,0


In [25]:
#Final model data verfication
model_data.isnull().sum()

retained            0
esent               0
eopenrate           0
eclickrate          0
avgorder            0
ordfreq             0
paperless           0
refill              0
doorstep            0
city_BOM            0
city_DEL            0
city_MAA            0
city_BLR            0
favday_Monday       0
favday_Tuesday      0
favday_Friday       0
favday_Thursday     0
favday_Wednesday    0
favday_Saturday     0
favday_Sunday       0
created_year        0
created_month       0
firstorder_year     0
firstorder_month    0
lastorder_year      0
lastorder_month     0
dtype: int64

In [26]:
# Prepare data SageMaker's Linear Learner
# Amazon SageMaker's Linear Learner container expects data in CSV data format. 
# Note that the first column must be the target variable and the CSV should not include headers. 

# move prediction field (ie, retained) to first column. 
# model_data2 = model_data.reindex(columns = ['retained','esent','eopenrate','eclickrate','avgorder','ordfreq','paperless','refill','doorstep',
#                                             'city_BOM','city_DEL','city_MAA','city_BLR','favday_Monday','favday_Tuesday','favday_Friday','favday_Thursday','favday_Wednesday','favday_Saturday','favday_Sunday',
#                                             'created_year','created_month','firstorder_year','firstorder_month','lastorder_year ','lastorder_month'])

#but since retained is the 1st column, this step is not needed and we can assign model_data2 = model_data
model_data2 = model_data
model_data2.head()

,retained,esent,eopenrate,eclickrate,avgorder,ordfreq,paperless,refill,doorstep,city_BOM,...,favday_Thursday,favday_Wednesday,favday_Saturday,favday_Sunday,created_year,created_month,firstorder_year,firstorder_month,lastorder_year,lastorder_month
0,0,29,100.000000,3.448276,14.52,0.000000,0,0,0,0.0,...,0.0,0.0,0.0,0.0,2012,8,2013,7,2013,7
1,1,95,92.631579,10.526316,83.69,0.181641,1,1,1,0.0,...,0.0,0.0,0.0,0.0,2010,11,2011,3,2014,0
2,0,0,0.000000,0.000000,33.58,0.059908,0,0,0,0.0,...,0.0,1.0,0.0,0.0,2010,9,2010,11,2011,6
3,0,0,0.000000,0.000000,54.96,0.000000,0,0,0,1.0,...,1.0,0.0,0.0,0.0,2010,9,2011,2,2011,2
4,1,30,90.000000,13.333333,111.91,0.008850,0,0,0,1.0,...,0.0,0.0,0.0,0.0,2010,10,2010,10,2013,0


# For best results, ensure your data is shuffled before training. Training with unshuffled data may cause training to fail.



In [27]:
#randomizw/shuffle the data so all the 1s aren't at the endmodel_data2 = np.random.shuffle(model_data2.values)

model_data2 = model_data2.sample(frac=1)

In [28]:
model_data2

,retained,esent,eopenrate,eclickrate,avgorder,ordfreq,paperless,refill,doorstep,city_BOM,...,favday_Thursday,favday_Wednesday,favday_Saturday,favday_Sunday,created_year,created_month,firstorder_year,firstorder_month,lastorder_year,lastorder_month
11326,1,33,0.000000,0.000000,52.22,0.050847,1,0,0,1.0,...,0.0,0.0,0.0,0.0,2011,4,2011,4,2011,6
28648,1,31,16.129032,3.225806,40.02,0.000000,1,1,0,1.0,...,0.0,0.0,0.0,0.0,2017,9,2017,9,2017,9
21865,1,18,61.111111,11.111111,47.95,0.043478,1,0,0,0.0,...,0.0,1.0,0.0,0.0,2013,11,2013,11,2014,0
5951,1,41,39.024390,2.439024,51.50,0.000000,0,0,0,0.0,...,0.0,0.0,1.0,0.0,2012,8,2013,11,2013,11
2558,1,45,13.333333,4.444444,42.62,0.006452,1,0,0,0.0,...,0.0,0.0,0.0,0.0,2011,5,2011,5,2013,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11070,1,45,0.000000,0.000000,233.59,0.000000,0,0,0,0.0,...,0.0,0.0,0.0,0.0,2011,4,2011,4,2011,4
4816,1,21,85.714286,0.000000,39.70,0.000000,0,0,0,0.0,...,0.0,0.0,0.0,1.0,2013,2,2013,3,2013,3
9516,0,0,0.000000,0.000000,58.16,0.000000,0,0,0,1.0,...,0.0,0.0,0.0,0.0,2011,4,2011,4,2011,4
10943,1,35,8.571429,2.857143,88.03,0.285714,0,0,0,0.0,...,0.0,0.0,0.0,1.0,2012,7,2012,8,2012,8


In [29]:
# remove the headers
model_data2.columns = range(model_data2.shape[1])   # Delete headers

model_data2

,0,1,2,3,4,5,6,7,8,9,...,16,17,18,19,20,21,22,23,24,25
11326,1,33,0.000000,0.000000,52.22,0.050847,1,0,0,1.0,...,0.0,0.0,0.0,0.0,2011,4,2011,4,2011,6
28648,1,31,16.129032,3.225806,40.02,0.000000,1,1,0,1.0,...,0.0,0.0,0.0,0.0,2017,9,2017,9,2017,9
21865,1,18,61.111111,11.111111,47.95,0.043478,1,0,0,0.0,...,0.0,1.0,0.0,0.0,2013,11,2013,11,2014,0
5951,1,41,39.024390,2.439024,51.50,0.000000,0,0,0,0.0,...,0.0,0.0,1.0,0.0,2012,8,2013,11,2013,11
2558,1,45,13.333333,4.444444,42.62,0.006452,1,0,0,0.0,...,0.0,0.0,0.0,0.0,2011,5,2011,5,2013,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11070,1,45,0.000000,0.000000,233.59,0.000000,0,0,0,0.0,...,0.0,0.0,0.0,0.0,2011,4,2011,4,2011,4
4816,1,21,85.714286,0.000000,39.70,0.000000,0,0,0,0.0,...,0.0,0.0,0.0,1.0,2013,2,2013,3,2013,3
9516,0,0,0.000000,0.000000,58.16,0.000000,0,0,0,1.0,...,0.0,0.0,0.0,0.0,2011,4,2011,4,2011,4
10943,1,35,8.571429,2.857143,88.03,0.285714,0,0,0,0.0,...,0.0,0.0,0.0,1.0,2012,7,2012,8,2012,8


# Separate data into train/validation/test data split

In [30]:
# The model will be trained on 70% of data, it will then be evaluated on 20% of data to give us an estimate 
# of the accuracy, and 10% will be held back as a final testing dataset which will be used later on.

# Compute split indices
train_end = int(0.7 * len(model_data2))
val_end = int(0.9 * len(model_data2))

# Split the DataFrame into train, validation, and test sets
train_data = model_data2.iloc[:train_end]
validation_data = model_data2.iloc[train_end:val_end]
test_data = model_data2.iloc[val_end:]

In [31]:
print(f"train_data: {len(train_data)}")
print(f"validation_data: {len(validation_data)}")
print(f"test_data: {len(test_data)}")

train_data: 21530
validation_data: 6152
test_data: 3076


In [32]:
train_data.head()

,0,1,2,3,4,5,6,7,8,9,...,16,17,18,19,20,21,22,23,24,25
11326,1,33,0.000000,0.000000,52.22,0.050847,1,0,0,1.0,...,0.0,0.0,0.0,0.0,2011,4,2011,4,2011,6
28648,1,31,16.129032,3.225806,40.02,0.000000,1,1,0,1.0,...,0.0,0.0,0.0,0.0,2017,9,2017,9,2017,9
21865,1,18,61.111111,11.111111,47.95,0.043478,1,0,0,0.0,...,0.0,1.0,0.0,0.0,2013,11,2013,11,2014,0
5951,1,41,39.024390,2.439024,51.50,0.000000,0,0,0,0.0,...,0.0,0.0,1.0,0.0,2012,8,2013,11,2013,11
2558,1,45,13.333333,4.444444,42.62,0.006452,1,0,0,0.0,...,0.0,0.0,0.0,0.0,2011,5,2011,5,2013,7


In [33]:
# train data to CSV
train_data.to_csv('churnpredictor-train.csv', index=False, header=False)
validation_data.to_csv('churnpredictor-validation.csv', index=False, header=False)
test_data.to_csv('churnpredictor-test.csv', index=False, header=False)

In [34]:
bucket

'sagemaker-us-east-2-806700449556'

In [35]:
prefix

'sagemaker/ChurnPredictor-linlearn'

# Copy data to S3 for SageMaker to access

In [36]:
boto3.Session().resource('s3').Bucket(bucket).Object(os.path.join(prefix, 'train/train.csv')).upload_file('churnpredictor-train.csv')
boto3.Session().resource('s3').Bucket(bucket).Object(os.path.join(prefix, 'validation/validation.csv')).upload_file('churnpredictor-validation.csv')
boto3.Session().resource('s3').Bucket(bucket).Object(os.path.join(prefix, 'test/test.csv')).upload_file('churnpredictor-validation.csv')

# Train the model using Linear Learner

In [37]:
# specify the ECR container location for Amazon SageMaker's implementation of Linear Learner

container = sagemaker.image_uris.retrieve(region=boto3.Session().region_name, framework="linear-learner")

In [38]:
# Then, because we're training with the CSV file format, we'll create s3_inputs that our training function can use 
# as a pointer to the files in S3, which also specify that the content type is CSV.
s3_input_train = sagemaker.inputs.TrainingInput(s3_data='s3://{}/{}/train'.format(bucket, prefix), content_type='text/csv')
s3_input_validation = sagemaker.inputs.TrainingInput(s3_data='s3://{}/{}/validation/'.format(bucket, prefix), content_type='text/csv')
s3_input_test = sagemaker.inputs.TrainingInput(s3_data='s3://{}/{}/test/'.format(bucket, prefix), content_type='text/csv')

In [39]:
s3_input_test

In [40]:
sess = sagemaker.Session()

artificat_output_location = 's3://{}/{}/output'.format(bucket, prefix)
print("The model artifact will be loaded to: ", artificat_output_location)

ll = sagemaker.estimator.Estimator(container,
                                    role, 
                                    instance_count=1, 
                                    instance_type='ml.m4.xlarge',
                                    output_path=artificat_output_location,
                                    sagemaker_session=sess)
ll.set_hyperparameters(optimizer = "sgd",
                       learning_rate = 0.01,
                       mini_batch_size = 50,
                       epochs = 30,
                       predictor_type = "binary_classifier")

ll.fit({'train': s3_input_train, 'validation': s3_input_validation, 'test': s3_input_test})

INFO:sagemaker:Creating training-job with name: linear-learner-2025-11-09-04-42-23-523


The model artifact will be loaded to:  s3://sagemaker-us-east-2-806700449556/sagemaker/ChurnPredictor-linlearn/output
2025-11-09 04:42:24 Starting - Starting the training job...
2025-11-09 04:42:38 Starting - Preparing the instances for training...
2025-11-09 04:43:05 Downloading - Downloading input data...
2025-11-09 04:43:30 Downloading - Downloading the training image......
2025-11-09 04:44:56 Training - Training image download completed. Training in progress....Docker entrypoint called with argument(s): train
Running default environment configuration script
[11/09/2025 04:45:18 INFO 139670507624256] Reading default configuration from /opt/amazon/lib/python3.8/site-packages/algorithm/resources/default-input.json: {'mini_batch_size': '1000', 'epochs': '15', 'feature_dim': 'auto', 'use_bias': 'true', 'binary_classifier_model_selection_criteria': 'accuracy', 'f_beta': '1.0', 'target_recall': '0.8', 'target_precision': '0.8', 'num_models': 'auto', 'num_calibration_samples': '10000000', 

# Model Evaluation and Tuning

In [41]:
from sagemaker.tuner import IntegerParameter, CategoricalParameter, ContinuousParameter, HyperparameterTuner

hyperparameter_ranges = {
    "learning_rate": ContinuousParameter(0.00001, 1.0),
    "mini_batch_size": IntegerParameter(16, 64)
}

In [42]:
objective_metric_name = "validation:binary_classification_accuracy"

In [48]:
tuner = HyperparameterTuner(ll,
                            hyperparameter_ranges=hyperparameter_ranges,
                            max_jobs=10,
                            max_parallel_jobs=3,
                            objective_metric_name=objective_metric_name)

In [49]:
tuner.fit({'train': s3_input_train, 'validation': s3_input_validation})

INFO:sagemaker:Creating hyperparameter tuning job with name: linear-learner-251109-0507


...............................................................................................................................................................!


In [ ]:
tuning_job_name = tuner.latest_tuning_job.job_name

In [ ]:
tuner = HyperparameterTuner.attach(tuning_job_name=tuning_job_name)
tuner.describe()["BestTrainingJob"]["FinalHyperParameterTuningJobObjectiveMetric"]["Value"]